# Local SRE – Fine-tune incident summary model (Unsloth + LoRA)
Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.

In [ ]:
%%capture
!pip install unsloth

Upload `incident_summaries.jsonl` using the Files panel (left sidebar), then run the next cell.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",  # small = fast on free Colab + runs on CPU laptop
    max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(          # LoRA: train small adapter matrices only
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="incident_summaries.jsonl", split="train")
ds = ds.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)})
print(len(ds)); print(ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,   # if you get an error here, change to: processing_class=tokenizer
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=1024,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, max_steps=60, learning_rate=2e-4,
        logging_steps=5, optim="adamw_8bit", seed=42,
        output_dir="outputs", report_to="none",
    ),
)
trainer.train()   # ~3-6 minutes on T4. Watch the loss go down.

In [ ]:
# Quick test inside Colab
FastLanguageModel.for_inference(model)
msgs = [{"role":"system","content":"You write incident summaries in our team style. Be short and factual."},
        {"role":"user","content":"Alert: OOMKilled on billing-api. Started 10:05 UTC, resolved after 25 min. Root cause: billing-api loaded too much data in memory and hit its 512Mi limit. Impact: jobs restarted 5 times. Fix: Disabled the new flag and raised memory limit to 1Gi. Owner: @platform-oncall."}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.2)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# Export to GGUF (q4_k_m = small, good quality). Takes ~5-10 min.
model.save_pretrained_gguf("sre_summary_gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh sre_summary_gguf

In [ ]:
# Download the .gguf file (about 0.8 GB)
import glob
from google.colab import files
gguf = [f for f in glob.glob("**/*.gguf", recursive=True) if "Q4_K_M" in f.upper()][0]
print(gguf)
files.download(gguf)